# MNIST train / finetune splits for perturbation holdout

This notebook prepares MNIST splits that mirror the transcriptomics setup:

- **Context** `c`: fixed (`c=0`), like a single donor.
- **Perturbation** `p`: digit label `0..9`.
- **Pretrain**: all digits except one held-out digit (default `8`).
- **Finetune**: only the held-out digit.

We load the official MNIST **train** (60k) and **test** (10k) IDX files, concatenate them
into one pool (70k images), then split that pool into pretrain vs finetune by digit. There is
no separate evaluation test set — visual checks compare generated vs real samples from these pools.

Outputs are written to `data_dir/mnist/processed/`:

- `pretrain.pt` — seen digits (`p != held_out`)
- `finetune.pt` — held-out digit only
- `manifest.json` — metadata and counts

In [ ]:
from __future__ import annotations

import json
import struct
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

_repo = Path.cwd()
while _repo != _repo.parent:
    if (_repo / "pyproject.toml").is_file() and (_repo / "src" / "trajectory_replay").is_dir():
        break
    _repo = _repo.parent
else:
    raise FileNotFoundError("Could not locate repo root from this notebook path.")

sys.path.insert(0, str(_repo.resolve()))

DATA_DIR = _repo / "data_dir" / "mnist"
OUT_DIR = DATA_DIR / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

HELD_OUT_DIGIT = 8
NORMALIZE = "zero_one"  # "zero_one" or "standard"
SUBSAMPLE_PER_DIGIT: int | None = None  # e.g. 2000 for quick runs; None = use all
SUBSAMPLE_SEED = 0

print(f"Repo root: {_repo}")
print(f"Raw MNIST dir: {DATA_DIR}")
print(f"Output dir: {OUT_DIR}")

In [ ]:
def read_idx_images(path: Path) -> np.ndarray:
    """Return uint8 array (N, H, W) from MNIST idx3-ubyte file."""
    with path.open("rb") as f:
        magic, n, rows, cols = struct.unpack(">IIII", f.read(16))
        if magic != 2051:
            raise ValueError(f"Bad magic for images {path}: {magic}")
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(n, rows, cols)


def read_idx_labels(path: Path) -> np.ndarray:
    """Return uint8 array (N,) from MNIST idx1-ubyte file."""
    with path.open("rb") as f:
        magic, n = struct.unpack(">II", f.read(8))
        if magic != 2049:
            raise ValueError(f"Bad magic for labels {path}: {magic}")
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(n)


def load_mnist_all(data_dir: Path) -> tuple[np.ndarray, np.ndarray]:
    """Load MNIST train + test IDX files and concatenate into one pool."""
    x_train = read_idx_images(data_dir / "train-images.idx3-ubyte")
    y_train = read_idx_labels(data_dir / "train-labels.idx1-ubyte")
    x_test = read_idx_images(data_dir / "t10k-images.idx3-ubyte")
    y_test = read_idx_labels(data_dir / "t10k-labels.idx1-ubyte")
    x = np.concatenate([x_train, x_test], axis=0)
    y = np.concatenate([y_train, y_test], axis=0)
    return x, y


x_u8, y = load_mnist_all(DATA_DIR)
x_u8 = x_u8[:, np.newaxis, :, :]
print(f"MNIST pool: {x_u8.shape} (train 60k + test 10k)")

In [ ]:
def normalize_pixels(x: np.ndarray, mode: str) -> np.ndarray:
    if mode == "zero_one":
        return x / 255.0
    if mode == "standard":
        x01 = x / 255.0
        return (x01 - 0.1307) / 0.3081
    raise ValueError(f"Unknown normalize mode: {mode}")


def make_cp_labels(digits: np.ndarray, *, context_index: int = 0) -> tuple[np.ndarray, np.ndarray]:
    c = np.full(len(digits), context_index, dtype=np.int64)
    p = digits.astype(np.int64)
    return c, p


def stratified_subsample(
    x: np.ndarray,
    c: np.ndarray,
    p: np.ndarray,
    *,
    max_per_stratum: int | None,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if max_per_stratum is None:
        return x, c, p
    rng = np.random.default_rng(seed)
    keep = []
    for digit in np.unique(p):
        idx = np.flatnonzero(p == digit)
        if len(idx) > max_per_stratum:
            idx = rng.choice(idx, size=max_per_stratum, replace=False)
        keep.append(idx)
    keep = np.sort(np.concatenate(keep))
    return x[keep], c[keep], p[keep]


def split_by_digit(
    x: np.ndarray,
    c: np.ndarray,
    p: np.ndarray,
    held_out: int,
) -> dict[str, tuple[np.ndarray, np.ndarray, np.ndarray]]:
    pre_mask = p != held_out
    ft_mask = p == held_out
    return {
        "pretrain": (x[pre_mask], c[pre_mask], p[pre_mask]),
        "finetune": (x[ft_mask], c[ft_mask], p[ft_mask]),
    }

In [ ]:
x = normalize_pixels(x_u8, NORMALIZE)
c, p = make_cp_labels(y)
x, c, p = stratified_subsample(
    x, c, p, max_per_stratum=SUBSAMPLE_PER_DIGIT, seed=SUBSAMPLE_SEED
)

splits = split_by_digit(x, c, p, HELD_OUT_DIGIT)
x_pre, c_pre, p_pre = splits["pretrain"]
x_ft, c_ft, p_ft = splits["finetune"]

print(
    f"pretrain: {len(x_pre):5d} images (digits != {HELD_OUT_DIGIT}) | "
    f"finetune: {len(x_ft):5d} images (digit {HELD_OUT_DIGIT})"
)

In [ ]:
def digit_counts(p: np.ndarray) -> dict[int, int]:
    vals, cnts = np.unique(p, return_counts=True)
    return {int(v): int(c) for v, c in zip(vals, cnts, strict=True)}


def plot_digit_grid(
    x: np.ndarray,
    p: np.ndarray,
    *,
    title: str,
    n_per_digit: int = 8,
    seed: int = 0,
) -> None:
    rng = np.random.default_rng(seed)
    digits = sorted(np.unique(p).tolist())
    n_cols = n_per_digit
    n_rows = len(digits)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(1.4 * n_cols, 1.4 * n_rows))
    if n_rows == 1:
        axes = np.expand_dims(axes, axis=0)
    for row, digit in enumerate(digits):
        idx = np.flatnonzero(p == digit)
        pick = idx if len(idx) <= n_cols else rng.choice(idx, size=n_cols, replace=False)
        for col in range(n_cols):
            ax = axes[row, col]
            ax.axis("off")
            if col < len(pick):
                img = x[pick[col]].reshape(28, 28)
                ax.imshow(img, cmap="gray", vmin=0.0, vmax=1.0)
            if col == 0:
                ax.set_ylabel(f"p={digit}", rotation=0, labelpad=28, va="center")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


print("Pretrain digit counts:", digit_counts(p_pre))
print(f"Finetune digit counts: {digit_counts(p_ft)}")

plot_digit_grid(x_pre, p_pre, title=f"Pretrain digits (held out {HELD_OUT_DIGIT})", seed=1)
plot_digit_grid(x_ft, p_ft, title=f"Finetune digit (p={HELD_OUT_DIGIT})", n_per_digit=16, seed=2)

In [ ]:
def save_split(path: Path, x: np.ndarray, c: np.ndarray, p: np.ndarray) -> None:
    torch.save(
        {"x": torch.from_numpy(x), "c": torch.from_numpy(c), "p": torch.from_numpy(p)},
        path,
    )


manifest = {
    "held_out_digit": HELD_OUT_DIGIT,
    "data_dim": 784,
    "image_shape": [28, 28],
    "n_contexts": 1,
    "n_perturbations": 10,
    "context_index": 0,
    "normalize": NORMALIZE,
    "subsample_per_digit": SUBSAMPLE_PER_DIGIT,
    "subsample_seed": SUBSAMPLE_SEED,
    "source": "mnist_train_test_concat",
    "mapping": {
        "context_c": "fixed dummy context (single donor analogue)",
        "perturbation_p": "digit label 0..9",
    },
    "pretrain": {
        "n": int(len(x_pre)),
        "digit_counts": digit_counts(p_pre),
    },
    "finetune": {
        "n": int(len(x_ft)),
        "digit_counts": digit_counts(p_ft),
    },
}

save_split(OUT_DIR / "pretrain.pt", x_pre, c_pre, p_pre)
save_split(OUT_DIR / "finetune.pt", x_ft, c_ft, p_ft)
with (OUT_DIR / "manifest.json").open("w") as f:
    json.dump(manifest, f, indent=2)

print(f"Wrote {OUT_DIR / 'pretrain.pt'}")
print(f"Wrote {OUT_DIR / 'finetune.pt'}")
print(f"Wrote {OUT_DIR / 'manifest.json'}")

## Next steps

See `experiments/mnist/README.md` for the training / finetuning pipeline plan.

Quick sanity check after running the scripts:

1. **Pretrain** should fit digits `0..7,9` well; digit `8` should fail.
2. **Conditional finetune** (`generative_replay`, `generative_field_replay`) should recover digit `8` while preserving seen digits.
3. **Unconditional finetune** (no new digit embedding) tests whether field replay still regularizes without a dedicated condition vector for the OOD digit.